# KNN

In [2]:
# 1. 导入必要的库。
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 2. 读入原始数据。使用pandas的read_csv读取训练集，进入DataFrame便于后续处理。
df = pd.read_csv(r'data/FBlocation/train.csv')

# 3. 数据空间过滤。只选取坐标x在1.0~1.25之间且y在2.5~2.75之间的数据，会让KNN计算时只在一个小区域聚焦，提高分类精度，同时减少样本量方便计算（KNN计算量与数据量密切相关）。
filtered_df = df[(df['x'] > 1.0) & (df['x'] < 1.25) & (df['y'] > 2.5) & (df['y'] < 2.75)]

# 4. 将 time 列转换为包含时间信息的新特征。KNN 不会自动提取时序特性，因此要手动加工为更有意义的特征：
# - datetime：将分钟数转为日期时间格式，便于分解
filtered_df['datetime'] = pd.to_datetime(filtered_df['time'], unit='m', origin='1970-01-01')
# - hour：每天的小时段，反映场所可能的使用高峰（如某些hour访问概率大），利于KNN更好地区分类别
filtered_df['hour'] = filtered_df['datetime'].dt.hour
# - weekday：用星期几替代绝对时间，提升KNN对周期性行为的识别能力
filtered_df['weekday'] = filtered_df['datetime'].dt.weekday
# - month：月度信息，有些地点可能存在季节性
filtered_df['month'] = filtered_df['datetime'].dt.month

# 5. 构建特征矩阵和标签：
#   - 特征集 X 中包括空间位置(x, y)、定位精度(accuracy)、和时间衍生特征(hour, weekday, month)
#   - KNN对所有特征都做“距离”利用，选用有意义的特征有助模型区分不同class
X = filtered_df[['x', 'y', 'accuracy', 'hour', 'weekday', 'month']]
y = filtered_df['place_id'] # 这里 place_id 是分类标签，KNN要找到与其“最相近”的邻居标签

# 6. 拆分训练集和测试集。KNN不训练参数，但需要有未见过的新样本检验泛化能力。test_size=0.2保持8:2分。
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. 特征标准化。
#   - 量纲指的是每个特征的单位及其取值范围，比如"x"和"y"是空间坐标，通常是米或类似长度单位；"accuracy"可能是误差范围（米），"hour"为小时（0~23），"weekday"为星期几（0~6），"month"为月份（1~12）。
#   - 如果变量的量纲（单位/数值范围）不同，KNN的距离计算中数值大的特征会对距离贡献更大，可能使模型忽略那些取值范围小但实际有区分作用的特征。
#   - 因此，需要对所有特征做统一的标准化，把它们的数值范围（通常转换为均值为0，方差为1）拉到同一标准，这样每个特征在“距离”比较时才有均等影响，避免某一特征因量纲大而主导计算结果。
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # 训练集标准化，fit+transform是规范操作
X_test_scaled = scaler.transform(X_test)         # 测试集仅transform，避免信息泄露

# 8. 构建KNN模型并训练。
#   - n_neighbors=5，为常用经验值，KNN的核心参数，表示“投票人数”
#   - fit方法会将训练集特征（X_train_scaled）和标签（y_train）分别存储在KNeighborsClassifier对象的
#     self._fit_X和self._y属性中。这样在进行预测时，KNN会利用这些存储的原始训练样本，通过距离计算查找最近邻，实现分类。
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# 9. 用KNN模型对测试集预测
#   - KNN本质就是读取存储的训练集特征，计算新样本与训练样本的距离，并投票分类
y_pred = knn.predict(X_test_scaled)

# 10. 模型评估
#    - 使用accuracy_score对预测标签与真实标签做精度统计
#    - KNN是监督分类算法，准确率是最常用评估指标，便于直观反映模型好坏
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')

Accuracy: 0.5175


# 网格搜索

In [3]:
from sklearn.model_selection import GridSearchCV

# 定义待调优的参数范围
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance']
}

# 构建GridSearchCV对象，五折交叉验证
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

# 输出最优参数和交叉验证得分
print("最优参数: ", grid_search.best_params_)
print(f"交叉验证最佳准确率: {grid_search.best_score_:.4f}")

# 用最优参数模型在测试集上评估
best_knn = grid_search.best_estimator_
y_pred_best = best_knn.predict(X_test_scaled)
best_accuracy = accuracy_score(y_test, y_pred_best)
print(f'最优KNN模型的测试集准确率: {best_accuracy:.4f}')

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


最优参数:  {'n_neighbors': 9, 'weights': 'distance'}
交叉验证最佳准确率: 0.5334
最优KNN模型的测试集准确率: 0.5375
